<a href="https://colab.research.google.com/github/Hecontra/Aqualimpia/blob/main/notebooks/Aqualimpia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from joblib import dump

df = pd.read_excel("dataset_set_A_aguas_residuales.xlsx")

print("Primeras filas del dataset:")
display(df.head())

print("Cantidad de filas y columnas:")
print(df.shape)

print("Columnas disponibles:")
print(df.columns.tolist())

print("Información del dataset:")
df.info()

Primeras filas del dataset:


,fecha_registro,planta,caudal_entrada_m3_d,DBO_entrada_mg_L,SST_entrada_mg_L,pH_entrada,energia_aeracion_kWh,lodos_generados_kg_d,DBO_salida_mg_L,cumplimiento_norma
0,2025-10-27,Planta Centro,6840.0,216.0,167.0,7.01,1764.8,560.0,38.0,0.0
1,2025-09-06,Planta Centro,6803.0,326.0,200.0,7.55,1409.3,654.0,32.1,0.0
2,2025-07-24,Planta Centro,4464.0,305.0,225.0,7.12,957.9,416.9,49.7,0.0
3,2025-10-12,Planta Centro,5598.0,248.0,312.0,7.83,1612.1,575.5,30.5,0.0
4,2025-09-11,Planta Centro,6231.0,323.0,207.0,7.10,1864.7,622.0,36.8,0.0


Cantidad de filas y columnas:
(208, 10)
Columnas disponibles:
['fecha_registro', 'planta', 'caudal_entrada_m3_d', 'DBO_entrada_mg_L', 'SST_entrada_mg_L', 'pH_entrada', 'energia_aeracion_kWh', 'lodos_generados_kg_d', 'DBO_salida_mg_L', 'cumplimiento_norma']
Información del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fecha_registro        200 non-null    object 
 1   planta                204 non-null    object 
 2   caudal_entrada_m3_d   204 non-null    float64
 3   DBO_entrada_mg_L      200 non-null    float64
 4   SST_entrada_mg_L      200 non-null    float64
 5   pH_entrada            200 non-null    float64
 6   energia_aeracion_kWh  200 non-null    float64
 7   lodos_generados_kg_d  200 non-null    float64
 8   DBO_salida_mg_L       200 non-null    float64
 9   cumplimiento_norma    200 non-null    float64
d

In [ ]:
df["fecha_registro"] = pd.to_datetime(df["fecha_registro"])

df["eficiencia_DBO_%"] = (
    (df["DBO_entrada_mg_L"] - df["DBO_salida_mg_L"])
    / df["DBO_entrada_mg_L"]
) * 100

df["eficiencia_DBO_%"] = df["eficiencia_DBO_%"].round(2)

df["estado_cumplimiento"] = df["cumplimiento_norma"].map({
    1: "Cumple",
    0: "No cumple"
})

display(df.head())

,fecha_registro,planta,caudal_entrada_m3_d,DBO_entrada_mg_L,SST_entrada_mg_L,pH_entrada,energia_aeracion_kWh,lodos_generados_kg_d,DBO_salida_mg_L,cumplimiento_norma,eficiencia_DBO_%,estado_cumplimiento
0,2025-10-27,Planta Centro,6840.0,216.0,167.0,7.01,1764.8,560.0,38.0,0.0,82.41,No cumple
1,2025-09-06,Planta Centro,6803.0,326.0,200.0,7.55,1409.3,654.0,32.1,0.0,90.15,No cumple
2,2025-07-24,Planta Centro,4464.0,305.0,225.0,7.12,957.9,416.9,49.7,0.0,83.70,No cumple
3,2025-10-12,Planta Centro,5598.0,248.0,312.0,7.83,1612.1,575.5,30.5,0.0,87.70,No cumple
4,2025-09-11,Planta Centro,6231.0,323.0,207.0,7.10,1864.7,622.0,36.8,0.0,88.61,No cumple


In [ ]:
resumen_planta = df.groupby("planta").agg(
    registros=("planta", "count"),
    caudal_promedio_m3_d=("caudal_entrada_m3_d", "mean"),
    DBO_entrada_promedio=("DBO_entrada_mg_L", "mean"),
    DBO_salida_promedio=("DBO_salida_mg_L", "mean"),
    eficiencia_promedio=("eficiencia_DBO_%", "mean"),
    energia_promedio=("energia_aeracion_kWh", "mean"),
    lodos_promedio=("lodos_generados_kg_d", "mean"),
    cumplimiento_promedio=("cumplimiento_norma", "mean")
).reset_index()

resumen_planta["cumplimiento_%"] = resumen_planta["cumplimiento_promedio"] * 100

resumen_planta = resumen_planta.round(2)

display(resumen_planta)

,planta,registros,caudal_promedio_m3_d,DBO_entrada_promedio,DBO_salida_promedio,eficiencia_promedio,energia_promedio,lodos_promedio,cumplimiento_promedio,cumplimiento_%
0,200,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Planta Centro,76,5046.43,285.19,35.90,87.51,1260.85,433.01,0.23,22.67
2,Planta Norte,72,5215.42,275.85,36.56,86.65,1299.31,450.47,0.17,16.90
3,Planta Sur,54,4684.52,278.80,36.06,87.10,1193.78,394.44,0.30,29.63
4,Planta Sur,1,54.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "DBO de salida promedio por planta",
        "Eficiencia promedio de remoción de DBO",
        "Caudal de entrada vs DBO de salida",
        "Cumplimiento normativo por planta"
    )
)

fig.add_trace(
    go.Bar(
        x=resumen_planta["planta"],
        y=resumen_planta["DBO_salida_promedio"],
        name="DBO salida promedio"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=resumen_planta["planta"],
        y=resumen_planta["eficiencia_promedio"],
        name="Eficiencia promedio"
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=df["caudal_entrada_m3_d"],
        y=df["DBO_salida_mg_L"],
        mode="markers",
        text=df["planta"],
        name="Caudal vs DBO salida"
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Bar(
        x=resumen_planta["planta"],
        y=resumen_planta["cumplimiento_%"],
        name="Cumplimiento normativo"
    ),
    row=2,
    col=2
)

fig.update_layout(
    title_text="Dashboard exploratorio de desempeño - AquaLimpia S.A.",
    height=750,
    width=1100,
    showlegend=False
)

fig.update_xaxes(title_text="Planta", row=1, col=1)
fig.update_yaxes(title_text="DBO salida promedio", row=1, col=1)

fig.update_xaxes(title_text="Planta", row=1, col=2)
fig.update_yaxes(title_text="Eficiencia (%)", row=1, col=2)

fig.update_xaxes(title_text="Caudal entrada (m3/d)", row=2, col=1)
fig.update_yaxes(title_text="DBO salida (mg/L)", row=2, col=1)

fig.update_xaxes(title_text="Planta", row=2, col=2)
fig.update_yaxes(title_text="Cumplimiento (%)", row=2, col=2)

fig.show()

fig.write_html("dashboard_aqualimpia.html")

print("Dashboard generado: dashboard_aqualimpia.html")

Dashboard generado: dashboard_aqualimpia.html


In [4]:
import os
import pandas as pd

os.makedirs("outputs", exist_ok=True)


df = pd.read_excel("dataset_set_A_aguas_residuales.xlsx")

df["fecha_registro"] = pd.to_datetime(df["fecha_registro"])

df["eficiencia_remocion_DBO_%"] = (
    (df["DBO_entrada_mg_L"] - df["DBO_salida_mg_L"]) / df["DBO_entrada_mg_L"]
) * 100

df["alerta_operativa"] = df.apply(
    lambda fila: "Alerta" if fila["cumplimiento_norma"] == 0 else "Sin alerta",
    axis=1
)

df["estado_cumplimiento"] = df["cumplimiento_norma"].map({
    1: "Cumple",
    0: "No cumple"
})


reporte_operaciones = df[[
    "fecha_registro",
    "planta",
    "caudal_entrada_m3_d",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "energia_aeracion_kWh",
    "lodos_generados_kg_d",
    "eficiencia_remocion_DBO_%",
    "alerta_operativa"
]]

reporte_operaciones.to_excel(
    "outputs/reporte_operaciones.xlsx",
    index=False
)

reporte_gestion_ambiental = df[[
    "fecha_registro",
    "planta",
    "DBO_salida_mg_L",
    "cumplimiento_norma",
    "estado_cumplimiento"
]]

reporte_gestion_ambiental.to_excel(
    "outputs/reporte_gestion_ambiental.xlsx",
    index=False
)

print("Archivos generados correctamente:")
print("outputs/reporte_operaciones.xlsx")
print("outputs/reporte_gestion_ambiental.xlsx")

Archivos generados correctamente:
outputs/reporte_operaciones.xlsx
outputs/reporte_gestion_ambiental.xlsx
